In [0]:
###

In [0]:
import pyspark.sql.functions as F 

In [0]:
###

In [0]:
checkpoint_time = F.current_timestamp()
checkpoint = f"/Volumes/sales_forecast_project/landing_data/checkpoint_volume/products/products_{checkpoint_time}"

In [0]:
from pyspark.sql.types import *
expected_schema = StructType(
 [StructField('Product_ID', StringType(), True),
  StructField('Product_Name', StringType(), True),
  StructField('Category', StringType(), True),
  StructField('Brand', StringType(), True),
  StructField('UOM', StringType(), True),
  StructField('Price', IntegerType(), True),
  StructField("ingestion_time", TimestampType(), True)
]                           
)

In [0]:
df = (
spark.readStream.format("cloudFiles").
option("cloudFiles.format","csv").
option("header","true").
option("cloudFiles.schemaLocation","/Volumes/sales_forecast_project/landing_data/schema_volume/products").
option("cloudFiles.schemaEvolutionMode","rescue")
.schema(expected_schema)
.load("/Volumes/sales_forecast_project/landing_data/landing_data_volume/Products")
.withColumn("ingestion_time", F.current_timestamp())
)

In [0]:
df.writeStream.format("delta").trigger(availableNow=True).option("checkpointLocation", checkpoint).toTable("sales_forecast_project.bronze.prodcuts_data")